# Validação completa

Executa todas as fontes de teste do pipeline, com e sem classificação, e junta os
números em uma tabela final (`resumo`).

**Com classificação** (real / inventada / incompleta):

| # | Fonte | Medida |
|---|---|---|
| A1 | Dev set (`data/txt`, 26 documentos) | métrica oficial |
| A2 | Holdout ZIP saneado (atual e `_v1`, se existir) | métrica oficial + diagnóstico |
| A3 | Base inteira (uma frase de citação por acórdão) | acertos por tribunal |
| A4 | Registro de regressão (`analise/casos_regressao.csv`) | casos que passam, falham ou continuam abertos |
| A5 | Estresse (ruído e troca de dígito sobre o dev set) | métrica oficial por cenário |
| A6 | Testes unitários (`tests/test_cenarios.py` e `tests/test_p0_*.py`) | aprovados / falhas |

**Sem classificação** (só detecção do trecho e do tipo):

| # | Fonte | Medida |
|---|---|---|
| B1 | `data/sinteticos_desafio` (atual e `_v1`, se existir) | precisão e recall do trecho, tipo certo, detecções sobre distratores |

Nada é gravado em `saida/`: as saídas intermediárias vão para uma pasta temporária.

In [16]:
import collections
import random
import subprocess
import sys
import tempfile
from pathlib import Path
import json
import pandas as pd

# raiz do projeto: sobe até achar data/goldenset_offsets.csv (roda de analise/ ou da raiz)
RAIZ = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "data" / "goldenset_offsets.csv").exists())
sys.path[:0] = [str(RAIZ / "src"), str(RAIZ / "tests")]

import eda_utils as u
import pipeline as p
from steps.avaliar_metrica import avaliar
from steps.salvar_submissao import gravar_saida
from validacao import estresse as est
from validacao import holdout_zip as h
from validacao import teste_base_inteira as t
import test_casos_regressao as reg

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 180)

DADOS = RAIZ / "data"
TMP = Path(tempfile.mkdtemp(prefix="validacao_"))   # saídas intermediárias (não vão para saida/)
con = u.conectar_base()
resolvedor = p.Resolvedor(con)
resumo = []   # (fonte, medida, valor): tabela final


def tabela_metrica(r):
    linhas = [{"nivel": "final", "score": r["score_final"]}]
    for n, v in r["niveis"].items():
        linhas.append({"nivel": n, "score": v["score"], "macro_f1": v["macro_f1"], "tau": v["tau"],
                       "bonus": v["b"], **{f"f1_{c}": x for c, x in v["f1_por_classe"].items()}})
    return pd.DataFrame(linhas).set_index("nivel")


# print("raiz:", RAIZ, "| temporários:", TMP)

# Com classificação

## A1. Dev set

Os 26 documentos de `data/txt` contra `data/goldenset_offsets.csv`, pela métrica oficial.
É o conjunto em que o pipeline foi ajustado, então o score é quase máximo.

In [2]:
arquivos = sorted((DADOS / "txt").glob("*.txt"))
docs_dev = [p.processar_documento(a.stem, p.ler_texto(a), resolvedor) for a in arquivos]
r_dev = avaliar(gravar_saida(docs_dev, TMP / "dev"), DADOS / "goldenset_offsets.csv")
resumo.append(("A1 dev set", "score final", r_dev["score_final"]))
tabela_metrica(r_dev)

26 documentos, 192 citações -> C:\Users\Camila\AppData\Local\Temp\validacao_g3qh1ymn\dev\json
C:\Users\Camila\AppData\Local\Temp\validacao_g3qh1ymn\dev\submission.csv: 26 documentos.


,score,macro_f1,tau,bonus,f1_real,f1_inventada,f1_incompleta
nivel,,,,,,,
final,1.099149,NaN,NaN,NaN,NaN,NaN,NaN
1,1.099170,1.0,0.0,0.099170,1.0,1.0,1.0
2,1.099138,1.0,0.0,0.099138,1.0,1.0,1.0


## A2. Holdout ZIP saneado

`holdout_zip.executar` sanea o ZIP em memória (ids numéricos, nomes `gen_nN_NNN`, alusões de
lei sem artigo removidas, reais duplicadas viradas `incompleta`), roda o pipeline e pontua.
Se existir o ZIP `_v1` (versão anterior), ele é avaliado também.

In [3]:
zips = {"atual": h.ZIP}
zip_v1 = h.ZIP.with_name(h.ZIP.stem + "_v1.zip")
if zip_v1.exists():
    zips["v1"] = zip_v1

holdout = {}
zip_original = h.ZIP
try:
    for nome, caminho in zips.items():
        h.ZIP = caminho   # executar() lê o caminho do módulo
        print("-- holdout", nome)
        holdout[nome] = h.executar(TMP / f"holdout_{nome}")
finally:
    h.ZIP = zip_original

for nome, (r, diag, saneado) in holdout.items():
    resumo.append((f"A2 holdout ZIP ({nome})", "score final", r["score_final"]))
    resumo.append((f"A2 holdout ZIP ({nome})", "classe errada", len(diag["classe_errada"])))
    resumo.append((f"A2 holdout ZIP ({nome})", "falsos negativos", len(diag["falsos_negativos"])))
    resumo.append((f"A2 holdout ZIP ({nome})", "falsos positivos", len(diag["falsos_positivos"])))

pd.concat({nome: tabela_metrica(r) for nome, (r, _, _) in holdout.items()})

-- holdout atual
26 documentos, 69 citações -> C:\Users\Camila\AppData\Local\Temp\validacao_g3qh1ymn\holdout_atual\saida\json
C:\Users\Camila\AppData\Local\Temp\validacao_g3qh1ymn\holdout_atual\saida\submission.csv: 26 documentos.
-- holdout v1
26 documentos, 55 citações -> C:\Users\Camila\AppData\Local\Temp\validacao_g3qh1ymn\holdout_v1\saida\json
C:\Users\Camila\AppData\Local\Temp\validacao_g3qh1ymn\holdout_v1\saida\submission.csv: 26 documentos.


score  macro_f1  tau     bonus   f1_real  f1_inventada  f1_incompleta
      nivel                                                                          
atual final  0.939751       NaN  NaN       NaN       NaN           NaN            NaN
      1      0.990579  0.903030  0.0  0.096950  0.800000      0.909091       1.000000
      2      0.914338  0.834035  0.0  0.096283  0.903226      0.695652       0.903226
v1    final  0.758455       NaN  NaN       NaN       NaN           NaN            NaN
      1      0.865931  0.791919  0.0  0.093459  0.800000      0.909091       0.666667
      2      0.704718  0.643500  0.0  0.095133  0.903226      0.727273       0.300000

In [4]:
# diagnóstico aproximado (IoU >= 0,5): quantas citações em cada categoria, por versão do ZIP
pd.DataFrame({nome: {k: len(v) for k, v in diag.items()} for nome, (_, diag, _) in holdout.items()})

,atual,v1
pareadas,59,47
falsos_negativos,8,20
falsos_positivos,10,8
classe_errada,1,2
id_errado,0,0


In [5]:
# cada erro do holdout, com o que era esperado e o que o pipeline produziu
erros = []
for nome, (_, diag, _) in holdout.items():
    for tipo_erro in ("falsos_negativos", "falsos_positivos", "classe_errada", "id_errado"):
        for item in diag[tipo_erro]:
            erros.append({"zip": nome, "erro": tipo_erro, "documento": item[0],
                          "trecho": item[1].replace(chr(10), " "), "detalhe": " -> ".join(map(str, item[2:]))})
pd.DataFrame(erros)

,zip,erro,documento,trecho,detalhe
0,atual,falsos_negativos,gen_n1_003,art. 818 da Consolidação das Leis do Trabalho,real
1,atual,falsos_negativos,gen_n1_009,Súmula Vinculante 10 do STF,real
2,atual,falsos_negativos,gen_n1_013,"Enunciado 92 do Fórum Nacional da Infância, aprovado em 2022",inventada
3,atual,falsos_negativos,gen_n2_017,Súm. 38/TST sobre alternância semanal,inventada
4,atual,falsos_negativos,gen_n2_020,"artigo 290, caput, do CPM",real
5,atual,falsos_negativos,gen_n2_023,art. 896 da C L T,real
6,atual,falsos_negativos,gen_n2_023,"Tema 1.284 do Supremo, sobre cooperativa de apoio logístico",inventada
7,atual,falsos_negativos,gen_n2_026,Tema 1.417/STJ — monitoramento glicêmico domiciliar,inventada
8,atual,falsos_positivos,gen_n1_002,"acórdão Min. Sérgio Kukina, 2018",incompleta
9,atual,falsos_positivos,gen_n1_002,artigo da Lei nº 14.133/2021,inventada


## A3. Base inteira

Para cada acórdão da base, uma frase de citação simples é passada pelo pipeline; vale se ele
extrai o trecho e resolve para o `id` certo. `duplicata_legitima` (`incompleta` é a resposta
certa) fica fora das contas de acerto.

In [6]:
medicao = t.executar(con)

por_tribunal = pd.DataFrame(
    [{"tribunal": trib, "ok": ok, "avaliaveis": total, "taxa": ok / total}
     for trib, (ok, total) in sorted(t.ok_por_tribunal(medicao).items())]
).set_index("tribunal")
for trib, linha in por_tribunal.iterrows():
    resumo.append((f"A3 base inteira ({trib})", "acertos / avaliáveis", f"{int(linha.ok)}/{int(linha.avaliaveis)}"))
resumo.append(("A3 base inteira", "extras na frase", medicao["extras"]))
resumo.append(("A3 base inteira", "previsões em distratores", f"{medicao['fp_distratores']}/{medicao['total_distratores']}"))
por_tribunal

,ok,avaliaveis,taxa
tribunal,,,
STF,196,200,0.980000
STJ,167,179,0.932961
STM,149,171,0.871345
TSE,110,119,0.924370
TST,120,176,0.681818


In [7]:
pd.Series(medicao["resultado"], name="registros").sort_values(ascending=False).to_frame()

,registros
ok,742
duplicata_legitima,137
nao_extraiu,97
incompleta,4
real_errado,2


In [8]:
# principais falhas: tribunal, desfecho, classe, caminho
pd.DataFrame([{"tribunal": k[0], "desfecho": k[1], "classe": k[2], "caminho": k[3], "n": n}
              for k, n in medicao["falhas"].most_common(20)])

,tribunal,desfecho,classe,caminho,n
0,TST,nao_extraiu,AIRR,-,19
1,TST,nao_extraiu,Ag-AIRR,-,15
2,STM,nao_extraiu,EMBARGOS INFRINGENTES E DE NULIDADE,-,14
3,TSE,duplicata_legitima,TRIBUNAL SUPERIOR ELEITORAL ACÓRDÃO AGRAVO RE,acordao_ambigua,12
4,TSE,duplicata_legitima,TRIBUNAL SUPERIOR ELEITORAL ACÓRDÃO EMBARGOS,acordao_ambigua,12
5,TST,duplicata_legitima,RR,acordao_ambigua,12
6,STM,duplicata_legitima,APELAÇÃO,acordao_ambigua,12
7,TST,nao_extraiu,RRAg,-,11
8,TSE,duplicata_legitima,TRIBUNAL SUPERIOR ELEITORAL ACÓRDÃO RECURSO O,acordao_ambigua,10
9,STM,duplicata_legitima,APELAÇÃO CRIMINAL,acordao_ambigua,10


## A4. Registro de regressão

Cada linha de `analise/casos_regressao.csv` é reexecutada. `regressao` deve passar;
`aberto` deve continuar falhando (se passar, a linha precisa ser atualizada); `pendente`
não é executado.

In [9]:
linhas = []
for caso in reg.carregar_casos():
    modo = caso["modo_teste"]
    problemas = []
    if modo == "pendente":
        situacao = "pendente"
    else:
        citacoes = p.processar_documento("regressao", caso["exemplo"], resolvedor)["citacoes"]
        problemas = reg.divergencias(caso, citacoes)
        if modo == "regressao":
            situacao = "ok" if not problemas else "FALHA (regrediu)"
        else:
            situacao = "aberto" if problemas else "PASSOU (atualizar a linha)"
    linhas.append({"exemplo": caso["exemplo"].replace(chr(10), " "), "modo": modo, "situacao": situacao,
                   "divergencias": "; ".join(problemas)})

casos = pd.DataFrame(linhas)
for situacao, n in casos["situacao"].value_counts().items():
    resumo.append(("A4 regressão", situacao, int(n)))
casos["situacao"].value_counts().to_frame("casos")

,casos
situacao,
ok,41
aberto,19
pendente,2


In [10]:
# o que não está ok
casos[casos["situacao"] != "ok"]

,exemplo,modo,situacao,divergencias
19,"Nos termos do art. 1º da Lei de Murphy, aplica-se.",aberto,aberto,"span: esperado conter 'Lei de Murphy', obtido 'art. 1º da Lei'"
21,"Conforme o AIRR 123456-78.2019.5.02.0001 do TST, decidiu-se.",aberto,aberto,"citações: esperado 1, obtido 0; nenhuma citação detectada"
22,Conforme os Embargos Infringentes e de Nulidade 12-34.2015.7.00.00...,aberto,aberto,"citações: esperado 1, obtido 0; nenhuma citação detectada"
23,"Conforme o REsp 1.741.784/PR e HC 123456 do STF, decidiu-se.",aberto,aberto,"citações: esperado 2, obtido 1"
24,"Conforme a Lei 8.078/1990 e Lei 13.105/2015, art. 5.",aberto,aberto,"citações: esperado 2, obtido 1"
25,"Conforme a Súmula 83 do STJ, aplica-se.",aberto,aberto,"classe: esperado real, obtido incompleta; id: esperado 1289710642,..."
26,"Conforme o Decreto 10.000/2019, art. 2.",aberto,aberto,"span: esperado conter 'Decreto', obtido '10.000/2019, art. 2'"
27,"Conforme a Resolução CNJ 400/2021, art. 2.",aberto,aberto,"span: esperado conter 'Resolução', obtido '400/2021, art. 2'"
28,A Lei de Murphy determina que tudo dá errado.,pendente,pendente,
31,"Conforme o acórdão do STF de 2024, rel. Ministro Luís Roberto Barr...",pendente,pendente,


## A5. Estresse

Perturba as citações do gabarito do dev set e mede de novo: `ruido` (abreviação, formato do
número, separador de UF, OCR, quebra de linha; a classe continua `real`) e `digito` (troca um
dígito do identificador; a classe esperada passa a `inventada`).

In [11]:
SEEDS = 3
metrica = est.carregar_modulo(DADOS / "kaggle_metric.py", "kaggle_metric")
gabarito = est.carregar_gabarito()
textos = {a.stem: p.ler_texto(a) for a in arquivos}

linhas = []
for cenario in ["original", "ruido", "digito"]:
    for seed in range(1 if cenario == "original" else SEEDS):
        documentos, solucao = est.montar_cenario(cenario, textos, gabarito, random.Random(seed))
        r = metrica.avaliar(solucao, est.predizer(documentos, resolvedor), row_id="documento_id")
        linhas.append({"cenario": cenario, "seed": seed, "score_final": r["score_final"],
                       **{f"score_n{n}": v["score"] for n, v in r["niveis"].items()},
                       **{f"tau_n{n}": v["tau"] for n, v in r["niveis"].items()}})

estresse = pd.DataFrame(linhas)
for cenario, g in estresse.groupby("cenario"):
    resumo.append((f"A5 estresse ({cenario})", "score final (média)", round(g["score_final"].mean(), 4)))
estresse

,cenario,seed,score_final,score_n1,score_n2,tau_n1,tau_n2
0,original,0,1.099149,1.099170,1.099138,0.0,0.0
1,ruido,0,1.089733,1.079422,1.094889,0.0,0.0
2,ruido,1,1.086500,1.069722,1.094889,0.0,0.0
3,ruido,2,1.088507,1.075725,1.094899,0.0,0.0
4,digito,0,1.098289,1.098275,1.098296,0.0,0.0
5,digito,1,1.098289,1.098275,1.098296,0.0,0.0
6,digito,2,1.098289,1.098275,1.098296,0.0,0.0


## A6. Testes unitários

Casos escritos à mão (CNJ, normativos, qualificadores, IoU e cenários do resolvedor), rodados
com `pytest` em um subprocesso (grupo `dev` do `pyproject.toml`).

In [12]:
linhas = []
for arq in ["test_cenarios.py", "test_p0_avaliacao_iou.py", "test_p0_cnj.py",
            "test_p0_normativos.py", "test_p0_qualificadores.py"]:
    proc = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", f"tests/{arq}"],
                          cwd=RAIZ, capture_output=True, text=True, encoding="utf-8", errors="replace")
    saida = proc.stdout.strip().splitlines() or proc.stderr.strip().splitlines()
    linhas.append({"arquivo": arq, "retorno": "ok" if proc.returncode == 0 else "FALHA",
                   "resumo": saida[-1] if saida else ""})

testes = pd.DataFrame(linhas)
resumo.append(("A6 testes unitários", "arquivos aprovados", f"{(testes['retorno'] == 'ok').sum()}/{len(testes)}"))
testes

,arquivo,retorno,resumo
0,test_cenarios.py,ok,"[32m[32m[1m24 passed[0m, [32m[1m3 subtests passed[0m[32m i..."
1,test_p0_avaliacao_iou.py,ok,[32m[32m[1m6 passed[0m[32m in 0.03s[0m[0m
2,test_p0_cnj.py,ok,"[32m[32m[1m32 passed[0m, [32m[1m77 subtests passed[0m[32m ..."
3,test_p0_normativos.py,ok,"[32m[32m[1m18 passed[0m, [32m[1m38 subtests passed[0m[32m ..."
4,test_p0_qualificadores.py,ok,"[32m[32m[1m16 passed[0m, [32m[1m44 subtests passed[0m[32m ..."


# Sem classificação

## B1. `data/sinteticos_desafio`

Documentos com anotação só de **trecho e tipo** (`LEI` ou `JUR`), sem real/inventada/incompleta,
e uma lista de **distratores** (textos que parecem citação mas não são). Mede:

- recall: anotações que algum candidato pareou com IoU >= 0,5;
- precisão: detecções do pipeline que pareiam com uma anotação;
- tipo: o `tipo` do pipeline bate com o da anotação (`LEI` → `lei`, `JUR` → `jurisprudencia`);
- detecções sobre distratores: o pipeline marcou como citação algo que é distrator.

Se existir `data/sinteticos_desafio_v1`, a versão anterior também é medida.

In [ ]:
TIPO = {"LEI": "lei", "JUR": "jurisprudencia"}


def ler_jsonl(caminho):
    with caminho.open(encoding="utf-8") as arq:
        return [json.loads(linha) for linha in arq if linha.strip()]


def avaliar_deteccao(pasta):
    docs = ler_jsonl(pasta / "documentos.jsonl")
    anot = collections.defaultdict(list)
    for a in ler_jsonl(pasta / "anotacoes.jsonl"):
        anot[a["documento_id"]].append(a)
    distr = collections.defaultdict(list)
    for d in ler_jsonl(pasta / "distratores.jsonl"):
        distr[d["documento_id"]].append(d)

    linhas = []
    for doc in docs:
        doc_id = doc["documento_id"]
        citacoes = p.processar_documento(doc_id, doc["texto"], resolvedor)["citacoes"]
        livres = list(anot[doc_id])
        tp = tipo_ok = fp = fp_distrator = 0
        for c in citacoes:
            span = (c["inicio"], c["fim"])
            melhor = max(livres, key=lambda a: h._iou(span, (a["inicio"], a["fim"])), default=None)
            if melhor and h._iou(span, (melhor["inicio"], melhor["fim"])) >= 0.5:
                livres.remove(melhor)
                tp += 1
                tipo_ok += c["tipo"] == TIPO[melhor["tipo"]]
            else:
                fp += 1
                fp_distrator += any(c["inicio"] < d["fim"] and d["inicio"] < c["fim"] for d in distr[doc_id])
        linhas.append({"documento_id": doc_id, "anotacoes": len(anot[doc_id]), "detectadas": len(citacoes),
                       "acertos": tp, "tipo_certo": tipo_ok, "perdidas": len(livres),
                       "falsos_positivos": fp, "fp_em_distrator": fp_distrator, "distratores": len(distr[doc_id])})
    return pd.DataFrame(linhas)


pastas = {"atual": DADOS / "sinteticos_desafio"}
if (DADOS / "sinteticos_desafio_v1").exists():
    pastas["v1"] = DADOS / "sinteticos_desafio_v1"

deteccao = {nome: avaliar_deteccao(pasta) for nome, pasta in pastas.items()}

linhas = []
for nome, df in deteccao.items():
    s = df.drop(columns="documento_id").sum()
    linhas.append({"versao": nome, "anotacoes": s.anotacoes, "detectadas": s.detectadas,
                   "recall": s.acertos / s.anotacoes, "precisao": s.acertos / s.detectadas,
                   "tipo_certo": s.tipo_certo / s.acertos, "fp_em_distrator": f"{s.fp_em_distrator}/{s.distratores}"})
    resumo.append((f"B1 sinteticos_desafio ({nome})", "recall do trecho", round(s.acertos / s.anotacoes, 3)))
    resumo.append((f"B1 sinteticos_desafio ({nome})", "precisão do trecho", round(s.acertos / s.detectadas, 3)))
    resumo.append((f"B1 sinteticos_desafio ({nome})", "detecções em distratores", f"{s.fp_em_distrator}/{s.distratores}"))
pd.DataFrame(linhas).set_index("versao")

,anotacoes,detectadas,recall,precisao,tipo_certo,fp_em_distrator
versao,,,,,,
atual,16,14,0.5625,0.642857,1.0,3/33
v1,16,14,0.5000,0.571429,1.0,3/33


In [14]:
# por documento: onde o pipeline perdeu anotações ou marcou algo a mais
pd.concat(deteccao, names=["versao"]).reset_index(level=0)

,versao,documento_id,anotacoes,detectadas,acertos,tipo_certo,perdidas,falsos_positivos,fp_em_distrator,distratores
0,atual,sint_desafio_001,2,2,2,2,0,0,0,3
1,atual,sint_desafio_002,2,1,1,1,1,0,0,3
2,atual,sint_desafio_003,3,4,2,2,1,2,0,0
3,atual,sint_desafio_004,2,0,0,0,2,0,0,4
4,atual,sint_desafio_005,0,1,0,0,0,1,1,9
5,atual,sint_desafio_006,0,1,0,0,0,1,1,8
6,atual,sint_desafio_007,4,3,2,2,2,1,1,3
7,atual,sint_desafio_008,3,2,2,2,1,0,0,3
0,v1,sint_desafio_001,2,2,2,2,0,0,0,3
1,v1,sint_desafio_002,2,1,1,1,1,0,0,3


# Resumo

In [15]:
pd.DataFrame(resumo, columns=["fonte", "medida", "valor"])

,fonte,medida,valor
0,A1 dev set,score final,1.099149
1,A2 holdout ZIP (atual),score final,0.939751
2,A2 holdout ZIP (atual),classe errada,1
3,A2 holdout ZIP (atual),falsos negativos,8
4,A2 holdout ZIP (atual),falsos positivos,10
5,A2 holdout ZIP (v1),score final,0.758455
6,A2 holdout ZIP (v1),classe errada,2
7,A2 holdout ZIP (v1),falsos negativos,20
8,A2 holdout ZIP (v1),falsos positivos,8
9,A3 base inteira (STF),acertos / avaliáveis,196/200
